# 查看修改与回退

预计阅读时间：10-15 分钟。

提交前先看修改，回退前先看历史，是 Git 最重要的习惯之一。

## 原理

Git 中常见的查看操作有三类：

| 命令 | 作用 |
| --- | --- |
| `git status` | 查看当前文件状态 |
| `git diff` | 查看工作区中尚未暂存的修改 |
| `git diff --staged` | 查看暂存区中准备提交的修改 |
| `git log` | 查看提交历史 |
| `git show` | 查看某次提交的具体内容 |

上一节已经介绍过 `git restore`。本节重点介绍另一类回退操作：`git reset`。

## 示例：查看修改

继续使用前面的 `git-principle-demo`：



In [ ]:
cd git-principle-demo
git switch main




查看修改时，重点是比较三个位置：工作区、暂存区（index）和当前提交。当前提交通常是 `HEAD` 对应的提交。

修改一个已经被 Git 跟踪的文件：



In [ ]:
echo 'print("debug")' >> app.py
git status
git diff




`git diff` 比较的是工作区和暂存区，因此显示的是工作区中尚未暂存的修改。

把修改加入暂存区后，再查看暂存区差异：



In [ ]:
git add app.py
git status
git diff --staged




`git diff --staged` 比较的是暂存区和当前提交，因此显示的是下一次提交将包含的修改。

## 示例：查看历史

先把刚才的修改提交：



In [ ]:
git commit -m "Add debug output"




查看最近几次提交：



In [ ]:
git log --oneline --graph --decorate -5




查看当前提交的具体内容：



In [ ]:
git show HEAD




`HEAD` 表示当前位置。通常情况下，`HEAD` 指向当前分支，当前分支再指向当前提交。`HEAD~1` 表示当前提交的上一个提交。

## reset 的基本理解

`git reset` 常用于把当前分支移动到另一个提交。可以先把它理解为：移动 `HEAD` 指向的当前分支。

```mermaid
flowchart TB
    subgraph before[reset 前]
        direction LR
        A1[commit A] --> B1[commit B] --> C1[commit C]
    end
    main1([main]) -.-> C1
    HEAD1{{HEAD}} -.-> main1

    subgraph after[git reset HEAD~1 后]
        direction LR
        A2[commit A] --> B2[commit B] --> C2[commit C]
    end
    main2([main]) -.-> B2
    HEAD2{{HEAD}} -.-> main2

    classDef commit fill:#eef2ff,stroke:#4f46e5,color:#111827;
    classDef branch fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#064e3b;
    classDef head fill:#fff7ed,stroke:#ea580c,stroke-width:2px,color:#7c2d12;
    class A1,B1,C1,A2,B2,C2 commit;
    class main1,main2 branch;
    class HEAD1,HEAD2 head;
```

图中，`main` 从 `commit C` 回到 `commit B`。`commit C` 不再是当前分支历史的一部分。

## reset 的三种模式

`reset` 的关键区别在于：除了移动当前分支外，是否同时修改暂存区和工作区。

| 命令 | 当前分支 | 暂存区 | 工作区 | 常见用途 |
| --- | --- | --- | --- | --- |
| `git reset --soft HEAD~1` | 回退 | 不变 | 不变 | 撤销提交，但保留已暂存状态 |
| `git reset --mixed HEAD~1` | 回退 | 回退 | 不变 | 撤销提交，并取消暂存，默认模式 |
| `git reset --hard HEAD~1` | 回退 | 回退 | 回退 | 彻底回到旧提交，丢弃修改 |

如果不写模式，默认是 `--mixed`：



In [ ]:
git reset HEAD~1




`--hard` 会丢弃工作区修改，执行前必须确认这些修改不再需要。

## 示例：撤销最近一次提交

如果刚刚提交后发现提交信息或提交内容不合适，可以用 `reset` 撤销最近一次提交，同时保留工作区文件：



In [ ]:
git reset --mixed HEAD~1
git status




这会让当前分支回到上一个提交，暂存区也回到上一个提交，但工作区保留刚才的文件内容。之后可以重新 `git add` 和 `git commit`。

## 练习

请在一个独立目录中练习，不要在本课程项目仓库中直接操作，避免误回退课程文件。

创建一个新的练习仓库，并准备两次提交：



In [ ]:
mkdir git-diff-reset-practice
cd git-diff-reset-practice
git init -b main
echo 'print("v1")' > app.py
git add app.py
git commit -m "Add app"
echo 'print("v2")' >> app.py
git add app.py
git commit -m "Update app"




然后完成：

1. 修改 `app.py`，运行 `git diff` 查看工作区修改；
2. 使用 `git add app.py` 后，运行 `git diff --staged` 查看暂存区修改；
3. 使用 `git log --oneline --graph --decorate -5` 查看历史；
4. 选做：使用 `git reset --mixed HEAD~1` 撤销最近一次提交，并观察 `git status`。